# SOLIDIFY: Interactive Phase-Field Solidification & Metallurgy Lab

**Author:** Frank Cai (Purdue University, ORCID `0009-0003-0041-1459`)  
**Live WebGPU Instrument:** [https://solidify.frankcai.dev/app/](https://solidify.frankcai.dev/app/)  
**In-App Guided Labs:** [https://solidify.frankcai.dev/app/#labs](https://solidify.frankcai.dev/app/#labs)  
**Companion Physics Monograph:** [https://solidify.frankcai.dev/science/](https://solidify.frankcai.dev/science/)  
**Source & License:** [github.com/fronkt/solidify](https://github.com/fronkt/solidify) (MIT License)

---

## How to Use This nanoHUB Module
1. **Launch a Guided Lab** using one of the seven direct WebGPU buttons below (`#lab=1` through `#lab=7`). Each link opens SOLIDIFY's in-app Guided Labs tracker with live physics checks and automatic paired data capture (runs locally on your GPU at 60 fps in Chrome, Edge, Safari 26+, or recent Firefox).
2. **Export or print your lab report** inside SOLIDIFY by clicking **`export lab ZIP`** (or **`print report`** for a light-mode PDF-ready summary) in the Guided Labs panel, or **ADVANCED -> Export run ZIP**. The ZIP archive contains `lab-report.json`, `run.json`, raw CSV series under `data/`, and `micrograph.png`.
3. **Analyze your data right here in Python:** pass your exported ZIP bytes into `load_lab_report(zip_bytes)` below (or run with the built-in verified dataset) to print your completed-step scorecard, seed, intervention flag, and captured comparison table, plus cooling-curve derivatives ($dT/dt$), Scheil segregation curves, and Hall–Petch $\sigma_y$ vs. $d^{-1/2}$ fits.

In [ ]:
import io, json, pathlib, zipfile
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import display, HTML
except ImportError:
    def HTML(s):
        return s
    def display(obj):
        pass

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

LABS = [
    (1, 'Lab 1: Dendrites, Symmetry & Calibration', 'https://solidify.frankcai.dev/app/#lab=1', 'Compare 4-fold, 6-fold, and seaweed growth, Karma-Rappel calibration, SDAS convergence, and XRAY Scheil segregation.'),
    (2, 'Lab 2: Alloys, Screening & Scheil', 'https://solidify.frankcai.dev/app/#lab=2', 'Audit A356, AZ91, and 1045 steel phase diagrams, test 118-element refusals, and run a controlled growth-restriction Q pair.'),
    (3, 'Lab 3: 3D Casting, Cooling & Niyama', 'https://solidify.frankcai.dev/app/#lab=3', 'Compare furnace vs. quench recalescence, refiner fade, 3D step-block sections, 2D-vs-3D stereology, and steel Niyama porosity.'),
    (4, 'Lab 4: Heat Treat & Hall-Petch Specs', 'https://solidify.frankcai.dev/app/#lab=4', 'Contrast Arrhenius coarsening clocks, Zener pinning to meet a Hall-Petch spec, and 3D SFE-controlled Sigma-3 annealing twins.'),
    (5, 'Lab 5: Cold Work, Recovery & Recrystallization', 'https://solidify.frankcai.dev/app/#lab=5', 'Probe the 8 J_b flat-front barrier in 3D heat treat, observe second-order recovery stall, and measure Avrami n across nucleation modes.'),
    (6, 'Lab 6: Directional Growth, Melt Flow & Selection', 'https://solidify.frankcai.dev/app/#lab=6', 'Contrast Bridgman G/V regimes, qualitative 2D LBM melt flow asymmetry, and 3D helical grain selector staging with [001] pole figures.'),
    (7, 'Lab 7: Inverse Process Design & Optimization', 'https://solidify.frankcai.dev/app/#lab=7', 'Compete against CMA-ES in 2D challenge mode, optimize and verify an ASTM G recipe, and run 3D grain-diameter inverse optimization.'),
]

rows = ''.join(
    f"<tr><td style='padding:8px 12px;font-weight:600'>{title}</td>"
    f"<td style='padding:8px 12px'>{desc}</td>"
    f"<td style='padding:8px 12px'><a href='{url}' target='_blank' style='padding:5px 12px;background:#111;color:#fff;border-radius:14px;text-decoration:none;white-space:nowrap'>Launch Lab {num} &rarr;</a></td></tr>"
    for num, title, url, desc in LABS
)
display(HTML(f"<table style='border-collapse:collapse;width:100%;font-family:sans-serif'>{rows}</table>"))

## Part 0: Guided Lab Report Loader (`lab-report.json` + `run.json`)

Pass the raw bytes (or file path) of an exported SOLIDIFY Run-Export ZIP into `load_lab_report(zip_bytes)` to verify `lab-report.json` (`format: "solidify-lab-report"`, `version: 1`) alongside `run.json`. Leaving `zip_bytes=None` runs against a built-in Lab 3 demonstration report so the full notebook executes cleanly without an upload.

In [ ]:
def _demo_lab_bundle():
    """Built-in representative Lab 3 GuidedLabReport + run.json summary."""
    report = {
        'format': 'solidify-lab-report',
        'version': 1,
        'labId': 'lab-3',
        'hashId': 3,
        'title': '3D Casting, Cooling & Niyama',
        'course': 'Foundry · Solidification Processing',
        'startedAt': '2026-10-01T14:00:00Z',
        'completedAt': '2026-10-01T14:32:10Z',
        'build': 'demo-v8.1',
        'seed': 12345678,
        'setupLink': 'https://solidify.frankcai.dev/app/#lab=3',
        'stepsCompleted': 5,
        'stepsSkipped': 0,
        'stepsTotal': 5,
        'anyIntervened': False,
        'steps': [
            {'id': 'recalescence-ratchet', 'title': 'Recalescence ratchet', 'status': 'completed', 'completedAt': '2026-10-01T14:07:12Z', 'captures': {}},
            {'id': 'refiner-fade', 'title': 'Pre-pour refiner fade', 'status': 'completed', 'completedAt': '2026-10-01T14:12:04Z', 'captures': {}},
            {'id': 'step-block-3d', 'title': '3D step-block section effect', 'status': 'completed', 'completedAt': '2026-10-01T14:19:45Z', 'captures': {}},
            {'id': 'stereology-bias', 'title': '2D vs. 3D stereology bias', 'status': 'completed', 'completedAt': '2026-10-01T14:24:30Z', 'captures': {}},
            {'id': 'steel-niyama', 'title': 'Steel shrinkage & Niyama map', 'status': 'completed', 'completedAt': '2026-10-01T14:32:10Z', 'captures': {}},
        ],
        'columns': [
            {'key': 'T_L_C', 'label': 'T_L', 'unit': 'C'},
            {'key': 'dTr_K', 'label': 'dT_r', 'unit': 'K'},
            {'key': 'fired', 'label': 'Sites fired'},
            {'key': 'dBar_um', 'label': 'd_bar', 'unit': 'um'},
            {'key': 'sigmaY_MPa', 'label': 'sigma_y', 'unit': 'MPa'},
        ],
        'table': [
            {'group': 'Part A', 'label': 'furnace (argon, 0 min)', 'values': {'T_L_C': 648.1, 'dTr_K': 3.2, 'fired': 142, 'dBar_um': 46.2, 'sigmaY_MPa': 30.0}},
            {'group': 'Part A', 'label': 'quench (argon, 0 min)', 'values': {'T_L_C': 648.0, 'dTr_K': 2.6, 'fired': 418, 'dBar_um': 27.4, 'sigmaY_MPa': 33.0}},
            {'group': 'Part A', 'label': 'air fade (45 min hold)', 'values': {'T_L_C': 648.1, 'dTr_K': 3.5, 'fired': 168, 'dBar_um': 42.1, 'sigmaY_MPa': 30.5}},
            {'group': 'Part B', 'label': 'step 1 (thinnest)', 'values': {'T_L_C': None, 'dTr_K': None, 'fired': None, 'dBar_um': 14.0, 'sigmaY_MPa': 38.2}},
            {'group': 'Part B', 'label': 'step 4 (thickest)', 'values': {'T_L_C': None, 'dTr_K': None, 'fired': None, 'dBar_um': 34.0, 'sigmaY_MPa': 31.7}},
        ],
    }
    run_meta = {
        'format': 'solidify-run',
        'version': 1,
        'build': 'demo-v8.1',
        'solver': 'kobayashi-3d',
        'mode': '3d',
        'grid': 128,
        'seed': 12345678,
        'material': {'key': 'al', 'name': 'aluminum · Al-Cu'},
        'datasets': [],
    }
    return report, run_meta


def _ascii_safe(val):
    if val is None:
        return '-'
    if isinstance(val, float):
        return f'{val:.2f}' if abs(val) < 1000 else f'{val:.3g}'
    s = str(val)
    for src_ch, dst_ch in (
        ('\u00b7', '/'), ('\u2013', '-'), ('\u2014', '-'), ('\u00b5', 'u'),
        ('\u03bc', 'u'), ('\u00b0', ' deg'), ('\u0394', 'd'), ('\u03bb', 'lambda'),
        ('\u03c3', 'sigma'), ('\u03a3', 'Sigma'), ('\u00b2', '^2'), ('\u00b3', '^3'),
        ('\u2080', '_0'), ('\u2081', '_1'), ('\u2082', '_2'), ('\u2083', '_3'),
        ('\u0304', ''), ('\u00b1', '+/-'), ('\u2248', '~'), ('\u2265', '>='), ('\u2264', '<='),
    ):
        s = s.replace(src_ch, dst_ch)
    return s.encode('ascii', 'replace').decode('ascii')


def load_lab_report(zip_bytes=None):
    """Parse lab-report.json and run.json from a SOLIDIFY Run-Export ZIP (or built-in demo)."""
    report = None
    run_meta = None
    source_label = 'Built-in demo lab-report.json'

    if zip_bytes is not None:
        if isinstance(zip_bytes, (str, pathlib.Path)):
            src = pathlib.Path(zip_bytes)
            zf_ctx = zipfile.ZipFile(src)
            source_label = f'Uploaded ZIP: {src.name}'
        elif isinstance(zip_bytes, io.BytesIO):
            zip_bytes.seek(0)
            zf_ctx = zipfile.ZipFile(zip_bytes)
            source_label = 'Uploaded Run-Export ZIP'
        else:
            zf_ctx = zipfile.ZipFile(io.BytesIO(bytes(zip_bytes)))
            source_label = 'Uploaded Run-Export ZIP'

        with zf_ctx as zf:
            names = set(zf.namelist())
            if 'run.json' in names:
                run_meta = json.loads(zf.read('run.json').decode('utf-8'))
            if 'lab-report.json' in names:
                report = json.loads(zf.read('lab-report.json').decode('utf-8'))
                if report.get('format') != 'solidify-lab-report' or report.get('version') != 1:
                    raise ValueError(
                        f"Unsupported lab-report.json schema: format={report.get('format')!r}, version={report.get('version')!r}"
                    )
    else:
        report, run_meta = _demo_lab_bundle()

    print(f'=== SOLIDIFY Guided Lab Summary ({source_label}) ===')
    if report is None:
        print('No lab-report.json found in ZIP (standard sandbox run export).')
        if run_meta is not None:
            seed = run_meta.get('seed')
            mat = (run_meta.get('material') or {}).get('name', 'unknown')
            print(f"Material:   {_ascii_safe(mat)} ({run_meta.get('mode', '?').upper()}, grid {run_meta.get('grid', '?')})")
            print(f"Seed:       {seed} (0x{int(seed):08x})" if seed is not None else 'Seed:       -')
        return {'report': None, 'run': run_meta, 'source': source_label}

    steps = report.get('steps') or []
    steps_total = report.get('stepsTotal', len(steps) or 5)
    steps_completed = report.get('stepsCompleted')
    if steps_completed is None:
        steps_completed = sum(
            1 for s in steps if s.get('status') == 'completed' or s.get('completed') is True
        )
    steps_skipped = report.get('stepsSkipped')
    if steps_skipped is None:
        steps_skipped = sum(1 for s in steps if s.get('status') == 'skipped')

    seed = report.get('seed')
    if seed is None and run_meta is not None:
        seed = run_meta.get('seed')
    seed_str = f'{int(seed)} (0x{int(seed):08x})' if seed is not None else '-'
    intervened = bool(report.get('anyIntervened', False))

    print(f"Lab:        {_ascii_safe(report.get('labId'))} | {_ascii_safe(report.get('title'))}")
    print(f'Completed:  {steps_completed}/{steps_total} steps (skipped: {steps_skipped})')
    print(f'Seed:       {seed_str}')
    print(f"Intervened: {intervened} ({'mid-pour edit recorded' if intervened else 'clean run'})")
    if run_meta is not None:
        mat_name = (run_meta.get('material') or {}).get('name', '-')
        mode = str(run_meta.get('mode', '-')).upper()
        grid = run_meta.get('grid', '-')
        solver = run_meta.get('solver', '-')
        n_ds = len(run_meta.get('datasets') or [])
        print(f'Run export: {_ascii_safe(mat_name)} | {mode} | grid {grid} | {_ascii_safe(solver)} | {n_ds} CSV dataset(s)')

    if steps:
        print('\nStep statuses:')
        for idx, st in enumerate(steps, start=1):
            status = st.get('status') or ('completed' if st.get('completed') else 'pending')
            tag = {'completed': 'DONE', 'skipped': 'SKIP', 'pending': 'PEND'}.get(status, status.upper())
            reason = f" ({st['skipReason']})" if st.get('skipReason') else ''
            title = st.get('title') or st.get('id', f'step-{idx}')
            print(f'  {idx:02d}. [{tag}] {_ascii_safe(title)}{reason}')

    table = report.get('table') or []
    if table:
        cols_spec = report.get('columns') or []
        if cols_spec:
            val_keys = [c['key'] for c in cols_spec]
            val_headers = [
                f"{_ascii_safe(c['label'])} ({_ascii_safe(c['unit'])})" if c.get('unit') else _ascii_safe(c['label'])
                for c in cols_spec
            ]
        else:
            val_keys = []
            for row in table:
                for k in (row.get('values') or {}).keys():
                    if k not in val_keys:
                        val_keys.append(k)
            val_headers = [_ascii_safe(k) for k in val_keys]

        has_group = any(row.get('group') for row in table)
        headers = (['Group'] if has_group else []) + ['Row'] + val_headers
        matrix = []
        for row in table:
            vals = row.get('values') or {}
            line = ([_ascii_safe(row.get('group'))] if has_group else []) + [_ascii_safe(row.get('label'))]
            line.extend(_ascii_safe(vals.get(k)) for k in val_keys)
            matrix.append(line)

        widths = [max(len(h), *(len(r[i]) for r in matrix)) for i, h in enumerate(headers)]
        fmt = '  '.join(f'{{:<{w}}}' if i < (2 if has_group else 1) else f'{{:>{w}}}' for i, w in enumerate(widths))
        print('\nCaptured comparison table:')
        print('  ' + fmt.format(*headers))
        print('  ' + '  '.join('-' * w for w in widths))
        for r in matrix:
            print('  ' + fmt.format(*r))

    return {'report': report, 'run': run_meta, 'source': source_label}


lab_summary = load_lab_report()

## Part 1: Cooling-Curve Thermal Analysis & Recalescence Extraction

Upload a SOLIDIFY **Run Export ZIP** using `ipywidgets.FileUpload` (if running interactively) or leave blank to analyze the built-in Al–4Cu Lab Mode cooling-curve sample below.

In [ ]:
def load_cooling_data(zip_bytes=None):
    """Load cooling curve (t, T_C, fs) from a SOLIDIFY Run-Export ZIP, or return verified sample."""
    if zip_bytes is not None:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            csvs = [n for n in zf.namelist() if n.endswith('.csv') and ('cool' in n.lower() or 'pour' in n.lower())]
            if csvs:
                raw = zf.read(csvs[0]).decode('utf-8').splitlines()
                header = [h.strip() for h in raw[0].split(',')]
                arr = np.genfromtxt(raw[1:], delimiter=',')
                t = arr[:, 0]
                T = arr[:, 1]
                fs = arr[:, 2] if arr.shape[1] > 2 else np.linspace(0, 1, len(t))
                return t, T, fs, f'Uploaded: {csvs[0]}'

    # Built-in representative Al-4Cu air-cooled pour (T_L = 648 C)
    t = np.linspace(0.0, 20.0, 240)
    T_L = 648.0
    T = np.empty_like(t)
    fs = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < 3.5:
            T[i] = 682.0 - 9.7 * ti
        elif ti < 5.2:
            s = (ti - 3.5) / 1.7
            T[i] = T_L - 8.2 * np.sin(0.5 * np.pi * s)
            fs[i] = 0.04 * s**2
        elif ti < 7.0:
            s = (ti - 5.2) / 1.8
            T[i] = (T_L - 8.2) + 4.9 * np.sin(0.5 * np.pi * s)
            fs[i] = 0.04 + 0.18 * s
        elif ti < 15.0:
            s = (ti - 7.0) / 8.0
            T[i] = (T_L - 3.3) - 24.0 * s**1.35
            fs[i] = 0.22 + 0.76 * s
        else:
            s = (ti - 15.0) / 5.0
            T[i] = (T_L - 27.3) - 42.0 * s
            fs[i] = min(1.0, 0.98 + 0.02 * s)
    return t, T, fs, 'Built-in sample (Al-4Cu air-cooled pour)'

t, T, fs, source_label = load_cooling_data()
dTdt = np.gradient(T, t)

# Extract nucleation nadir T_N and recalescence peak
mask_nuc = (t > 2.5) & (t < 8.0)
i_nadir = np.where(mask_nuc)[0][np.argmin(T[mask_nuc])]
T_N = T[i_nadir]
mask_rec = (t > t[i_nadir]) & (t < t[i_nadir] + 3.5)
i_rec = np.where(mask_rec)[0][np.argmax(T[mask_rec])]
T_rec = T[i_rec]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
ax1.plot(t, T, color='#2a78d6', lw=2, label='Melt T(t)')
ax1.scatter([t[i_nadir], t[i_rec]], [T_N, T_rec], color='#eb6834', zorder=5)
ax1.annotate(f'T_N = {T_N:.1f} C', (t[i_nadir], T_N), xytext=(8, -14), textcoords='offset points')
ax1.annotate(f'Recalescence +{T_rec - T_N:.1f} K', (t[i_rec], T_rec), xytext=(8, 6), textcoords='offset points')
ax1.set_ylabel('Temperature (C)')
ax1.set_title(f'Cooling-Curve Thermal Analysis ({source_label})')
ax1.legend()

ax2.plot(t, dTdt, color='#0f8a60', lw=1.8, label='dT/dt')
ax2.axhline(0, color='#666', ls='--', lw=1)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('dT/dt (K/s)')
ax2.legend()
plt.tight_layout()
plt.show()

## Part 2: Gulliver–Scheil Segregation & Alloy Phase Boundaries

Compare equilibrium solidification (Lever Rule) against non-equilibrium dendritic freezing (Gulliver–Scheil) for SOLIDIFY's A356 (`Al–7.0Si`) and AZ91 (`Mg–9.0Al`) benchmarks.

In [ ]:
def scheil_curve(Tm, Tinv, Csm, Cinv, C0):
    k = Csm / Cinv
    m = (Tinv - Tm) / Cinv
    f_eut_scheil = (C0 / Cinv) ** (1.0 / (1.0 - k))
    f_s_max = 1.0 - f_eut_scheil
    fs_grid = np.linspace(0.0, f_s_max, 200)
    T_scheil = Tm + m * C0 * (1.0 - fs_grid) ** (k - 1.0)
    f_eut_lever = max(0.0, (C0 - Csm) / (Cinv - Csm))
    return fs_grid, T_scheil, f_eut_lever, f_eut_scheil

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
cases = [
    ('A356 (Al-7.0Si)', 660.35, 577.0, 1.65, 12.6, 7.0),
    ('AZ91 (Mg-9.0Al)', 650.0, 437.0, 12.9, 32.3, 9.0),
]
for ax, (name, Tm, Tinv, Csm, Cinv, C0) in zip(axes, cases):
    fs_g, T_g, fL, fS = scheil_curve(Tm, Tinv, Csm, Cinv, C0)
    ax.plot(fs_g, T_g, color='#2a78d6', lw=2, label='Gulliver-Scheil')
    ax.plot([fs_g[-1], 1.0], [Tinv, Tinv], color='#eb6834', lw=2.2, label=f'Eutectic ({fS*100:.1f}%)')
    ax.axhline(Tinv, color='#888', ls=':', lw=1)
    ax.set_title(f'{name}\nLever: {fL*100:.0f}% | Scheil: {fS*100:.1f}%')
    ax.set_xlabel('Solid fraction f_s')
    ax.set_ylabel('Temperature (C)')
    ax.set_xlim(0, 1.02)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Part 3: Hall–Petch Regression & Verification Scorecard

Analyze your Lab 3 step-block or Lab 4 heat-treatment grain sizes ($\bar{d}$ in $\mu\text{m}$) on a Hall–Petch plot ($\sigma_y = \sigma_0 + k_{\text{HP}} d^{-1/2}$). When `lab_summary` contains `dBar_um` rows from an uploaded Lab 3 or Lab 4 ZIP, those values can be plotted directly.

In [ ]:
# Representative Al-4Cu Hall-Petch constants (sigma_0 = 20 MPa, k_HP = 0.068 MPa m^1/2)
sigma0, k_hp = 20.0, 0.068
# Pull captured dBar_um rows from lab_summary when present, else use representative values:
captured_d = [
    float(r['values']['dBar_um'])
    for r in ((lab_summary.get('report') or {}).get('table') or [])
    if isinstance((r.get('values') or {}).get('dBar_um'), (int, float))
]
d_um = np.array(sorted(captured_d) if len(captured_d) >= 2 else [12.0, 14.0, 20.1, 34.0, 48.6])
sigma_y = sigma0 + k_hp / np.sqrt(d_um * 1e-6)

fig, ax = plt.subplots(figsize=(6.8, 3.8))
x = 1.0 / np.sqrt(d_um)
ax.plot(x, sigma_y, 'o-', color='#2a78d6', lw=2, ms=6)
for xi, yi, di in zip(x, sigma_y, d_um):
    ax.annotate(f'{di:.1f} um', (xi, yi), xytext=(4, 5), textcoords='offset points', fontsize=8)
ax.set_xlabel(r'Inverse square-root grain diameter $d^{-1/2}$ ($\mu\mathrm{m}^{-1/2}$)')
ax.set_ylabel(r'Hall-Petch yield strength $\sigma_y$ (MPa)')
ax.set_title('Al-4Cu Grain-Size Strengthening (No Precipitate Aging)')
plt.tight_layout()
plt.show()

print('Verification summary (SOLIDIFY v8 calibrated benchmarks):')
for label, val, ref, err in [
    ('2D tip velocity V d0 / D', '0.01679', '0.0170 (Karma-Rappel 1998)', '1.2%'),
    ('2D parabolic tip radius rho / d0', '28.8', '27.6 (Tong et al. 2001)', '4.4%'),
    ('Gibbs-Thomson critical radius R*', '22.21 W0', '22.10 W0', '0.5%'),
    ('Homogenization eigenmode decay', 'rel err', 'exact discrete DCT-II', '9e-7'),
]:
    print(f'  - {label:34s}: {val:10s} vs {ref:28s} ({err})')